In [1]:
!python3 -m pip install chromadb

  Using cached pybase64-1.4.3-cp311-cp311-macosx_11_0_arm64.whl.metadata (8.7 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached bcrypt-5.0.0-cp39-abi3-macosx_10_12_universal2.whl.metadata (10 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached httptools-0.7.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (3.5 kB)
  Using cached uvloop-0.22.1-cp311-cp311-macosx_10_9_universal2.whl.metadata (4.9 kB)
  Using cached watchfiles-1.1.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 10.2 MB/s  0:00:02 eta 0:00:01
Using cached bcrypt-5.0.0-cp39-abi3-macosx_10_12_universal2.whl (495 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 9.4 M

In [2]:
import chromadb

In [3]:
chroma_client = chromadb.Client()

In [4]:
collection = chroma_client.create_collection(name="my_collection")

In [5]:
collection.add(
    ids=["id1", "id2"],
    documents=[
        "This is a document about pineapple",
        "This is a document about stock prices"
    ]
)

In [6]:
collection.peek()

{'ids': ['id1', 'id2'],
 'embeddings': array([[-7.09367776e-03,  6.55461624e-02, -1.16606196e-02,
          8.64458457e-02, -3.95435914e-02,  5.80052696e-02,
         -2.40756404e-02,  1.49505725e-02,  9.91733186e-03,
         -2.10740156e-02,  8.10110718e-02,  3.47032622e-02,
         -2.12536771e-02, -1.33664217e-02, -3.32268812e-02,
          2.06374619e-02,  3.50651285e-03,  3.84415351e-02,
         -1.53143276e-02,  7.31229410e-02,  5.14781326e-02,
          1.06985033e-01, -4.18041870e-02,  3.10128462e-02,
         -1.11353546e-02, -1.37998248e-02, -2.21655089e-02,
         -6.40951321e-02, -2.69081928e-02, -1.26517555e-02,
         -4.96102124e-03,  5.54248542e-02,  1.17435016e-01,
          5.17347082e-02, -5.17161973e-02, -3.09536681e-02,
          1.01460904e-01, -6.95117563e-02,  1.60278007e-01,
          4.23167050e-02, -2.92910151e-02, -6.58737263e-03,
          4.21532951e-02,  1.74003243e-02,  1.13289189e-02,
          5.93366995e-02, -6.98440149e-02,  5.52195264e-03,
  

In [11]:
results = collection.query(
    query_texts=["This is a document about pineapple"], # Chroma will embed this for you
    n_results=2 # how many results to return
)
results

{'ids': [['id1', 'id2']],
 'embeddings': None,
 'documents': [['This is a document about pineapple',
   'This is a document about stock prices']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None]],
 'distances': [[0.0, 1.1876788139343262]]}

In [13]:
!python3 -m pip install PyPDF2

  Using cached pypdf2-3.0.1-py3-none-any.whl.metadata (6.8 kB)
Using cached pypdf2-3.0.1-py3-none-any.whl (232 kB)

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [14]:
# pip install pypdf sentence-transformers chromadb

import chromadb
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer

# 1. Read PDF text
reader = PdfReader("AIML.pdf")
pages = [page.extract_text() for page in reader.pages]
full_text = " ".join(pages)

# 2. Simple chunking
chunks = [full_text[i:i+500] for i in range(0, len(full_text), 500)]

# 3. Embed chunks
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks).tolist()

# 4. Store in Chroma
# Inline Memory
#client = chromadb.Client()
client = chromadb.PersistentClient(path="./vector_db")
#Persistent Client
coll = client.create_collection(name="courses")

coll.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    metadatas=[{"source": "AIML.pdf"} for _ in chunks]
)

print("Stored", len(chunks), "chunks from AIML.pdf")


/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 13533.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Stored 25 chunks from AIML.pdf


In [15]:
query = "Can you list highlights of AIML course"
res = coll.query(query_texts=[query], n_results=1)

top_doc = res["documents"][0][0]
top_dist = res["distances"][0][0]

print("Best Match:\n")
print(top_doc)   # full text of the document
print(f"\nDistance: {top_dist:.4f}")

Best Match:

Artificial
 
Intelligence
 
&
 
Machine
 
Learning
 
–
 
AIML
 
(
 
Empowering
 
the
 
Next
 
Generation
 
of
 
AI
 
Engineers
 
)
 
AI/ML
 
Training
 
(AIML)
 
program
 
transforms
 
learners
 
into
 
job-ready
 
AI/ML
 
engineers.
 
Through
 
hands-on
 
projects
 
and
 
real-world
 
applications,
 
it
 
takes
 
you
 
from
 
core
 
fundamentals
 
to
 
advanced
 
AI/ML
 
skills
 
that
 
employers
 
value.
 
Industry-Ready
 
AI
 
Skills:
 
 
 
🤖
 
Python
 
for
 
AI/ML:
 
Master
 
AI-focused
 
Pyt

Distance: 1.1135
